# Milestone 8 — Supervised fine-tuning

## Goal

Test whether supervised fine-tuning teaches Ministral-3B transferable Text-to-SQL behavior beyond the selected Milestone 5 context-engineered baseline.

This notebook follows one controlled path:

1. verify and upload the frozen SFT artifacts;
2. submit and monitor exactly one conservative SFT job;
3. verify the deployed tuned model;
4. evaluate it on the 12-question `working_dev` gate with the same inference contract as the untuned model;
5. compare the saved runs deterministically; and
6. decide whether the tuned model earned a frozen evaluation.

## Outcome

The Azure training job succeeded, but SFT v1 did **not** improve working-set accuracy. Both the untuned and tuned models passed the same four questions. Executability increased from 8/12 to 9/12, while completion-token usage increased substantially.

**Decision:** SFT v1 did not pass the model-selection gate, so the protected 24-question frozen evaluation was not run. This is a valid negative experimental result: successful training does not imply successful adaptation.


## 1. Experimental contract

| Control | Value |
|---|---|
| Base model | Ministral-3B, Azure catalog version 1 |
| Adaptation | Supervised fine-tuning, Global Standard |
| Training / validation | 222 / 85 records |
| Epochs / seed | 2 / 42 |
| Batch size / learning-rate multiplier | Automatic / automatic |
| Selection split | 12-question `working_dev` |
| Protected split | 24-question `frozen_eval` |
| Inference contract | Same context, instructions, output normalization, and evaluator as Milestone 5 |
| Attempts | One per question; no repair or correctness retry |

Two epochs were intentionally conservative because the dataset is small. The frozen financial questions were excluded from training and model selection.

The action cells below are guarded. Rerunning the notebook does not upload files, create another training job, or call a deployed model unless the corresponding flag is explicitly enabled.


In [ ]:
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter
from urllib.parse import urlparse
import csv
import json
import os
import sqlite3

from dotenv import load_dotenv

from domain_adapt.context_engineering import (
    load_context_engineering_config,
    request_context_engineered_sql,
)
from domain_adapt.evaluation import compare_sql_results
from domain_adapt.evaluation_analysis import (
    compare_runs,
    index_results,
    summarize_results,
)
from domain_adapt.finch import make_reference_compatible


def find_repository_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root")


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_json_atomic(path: Path, value: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_suffix(path.suffix + ".tmp")
    temporary_path.write_text(
        json.dumps(value, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(path)


REPOSITORY_ROOT = find_repository_root(Path.cwd())
DATA_DIR = REPOSITORY_ROOT / "data"
TRAINING_DIR = DATA_DIR / "training"
RUN_DIR = DATA_DIR / "runs" / "fine_tuning"

load_dotenv(REPOSITORY_ROOT / ".env", override=True)

EXPECTED_CONFIG_SHA256 = (
    "574a775a2c91826a4d0073f78e84c40c3c4f5b6fbb61f656bd63f307252cdf68"
)
EXPECTED_ARTIFACTS = {
    "canonical": "7e2b0aeb4b5a00ba1b05665d803be7f5ea3b52cdedd87b5958e354c959eaa1a7",
    "train_jsonl": "0b49016759ab6c07d16564f0d78a0ffefbac1b1eb0d2760b4fb6b3cbbcff65f8",
    "validation_jsonl": "503be8d18732b1e3ff2df8e4e337af6ab8db971617c2fc49ed139f7a58fab25a",
    "manifest": "9c98a0117ecb6003eda2a62aa93fab3f1bc464338d2d4cbb59f2240f7eb0a500",
}

print("Repository root:", REPOSITORY_ROOT)


## 2. Verify the frozen training inputs

Fine-tuning begins from immutable, versioned artifacts produced in Milestone 7. Hash validation prevents an accidental edit from silently changing the experiment. The JSONL validator checks the required `system → user → assistant` message structure without making a network request.


In [ ]:
files_by_hash = {
    sha256_file(path): path
    for path in TRAINING_DIR.rglob("*")
    if path.is_file()
}

resolved_artifacts = {}
for name, expected_hash in EXPECTED_ARTIFACTS.items():
    artifact_path = files_by_hash.get(expected_hash)
    if artifact_path is None:
        raise FileNotFoundError(
            f"Could not find {name} with SHA-256 {expected_hash}"
        )
    resolved_artifacts[name] = artifact_path


def validate_sft_jsonl(path: Path, expected_records: int) -> int:
    assert not path.read_bytes().startswith(b"\xef\xbb\xbf")
    count = 0
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            messages = json.loads(line).get("messages")
            roles = tuple(message.get("role") for message in messages)
            assert roles == ("system", "user", "assistant"), (
                f"Unexpected roles at line {line_number}: {roles}"
            )
            assert all(
                isinstance(message.get("content"), str)
                and message["content"].strip()
                for message in messages
            )
            count += 1
    assert count == expected_records
    return count


training_records = validate_sft_jsonl(
    resolved_artifacts["train_jsonl"], 222
)
validation_records = validate_sft_jsonl(
    resolved_artifacts["validation_jsonl"], 85
)

project_endpoint = os.environ["AZURE_AI_PROJECT_ENDPOINT"].strip()
assert "domain-adapt-project-resource" in project_endpoint

print("Fine-tuning input preflight passed.")
print("Project endpoint host:", urlparse(project_endpoint).netloc)
print("Training records:", training_records)
print("Validation records:", validation_records)
print("All expected hashes matched; no network request made.")


## 3. Upload once and record a receipt

Uploading makes the two verified JSONL files available to Azure; it does not start training. The receipt binds Azure file IDs to local hashes. Leave `UPLOAD_FILES = False` after the first successful upload.


In [ ]:
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential


project_client = AIProjectClient(
    endpoint=project_endpoint,
    credential=DefaultAzureCredential(),
)
openai_client = project_client.get_openai_client()

RUN_DIR.mkdir(parents=True, exist_ok=True)
upload_receipt_path = RUN_DIR / "ministral_3b_sft_v1_upload.json"
UPLOAD_FILES = False


def upload_fine_tuning_file(path: Path):
    with path.open("rb") as handle:
        uploaded = openai_client.files.create(
            file=handle,
            purpose="fine-tune",
        )
    processed = openai_client.files.wait_for_processing(uploaded.id)
    if processed.status != "processed":
        raise RuntimeError(f"{path.name}: {processed.status}")
    return processed


if upload_receipt_path.exists():
    upload_receipt = json.loads(
        upload_receipt_path.read_text(encoding="utf-8")
    )
    print("Using existing upload receipt:", upload_receipt_path.name)
elif UPLOAD_FILES:
    train_file = upload_fine_tuning_file(
        resolved_artifacts["train_jsonl"]
    )
    validation_file = upload_fine_tuning_file(
        resolved_artifacts["validation_jsonl"]
    )
    upload_receipt = {
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "project_endpoint_host": urlparse(project_endpoint).netloc,
        "training": {
            "sha256": EXPECTED_ARTIFACTS["train_jsonl"],
            "records": 222,
            "azure_file_id": train_file.id,
            "azure_status": train_file.status,
        },
        "validation": {
            "sha256": EXPECTED_ARTIFACTS["validation_jsonl"],
            "records": 85,
            "azure_file_id": validation_file.id,
            "azure_status": validation_file.status,
        },
    }
    save_json_atomic(upload_receipt_path, upload_receipt)
    print("Uploaded verified artifacts and saved receipt.")
else:
    upload_receipt = None
    print("Upload skipped. Enable UPLOAD_FILES only for a first upload.")


## 4. Freeze the job contract and submit once

The contract captures every consequential training choice before submission. The job-creation cell refuses to create a second job when a receipt already exists. After the historical run, `SUBMIT_FINE_TUNING_JOB` must remain `False`.


In [ ]:
EXPECTED_CONTRACT_SHA256 = (
    "a8b01286ec4a0c39b581b29aea23e04cb57e7d1548126856740e6e4e5d02d5e1"
)
job_receipt_path = RUN_DIR / "ministral_3b_sft_v1_job.json"

if upload_receipt is not None:
    job_contract = {
        "experiment_name": "ministral_3b_sft_v1",
        "base_model": "Ministral-3B",
        "azure_catalog_version": "1",
        "customization_method": "supervised",
        "training_type": "GlobalStandard",
        "training_file_id": upload_receipt["training"]["azure_file_id"],
        "training_sha256": upload_receipt["training"]["sha256"],
        "training_records": 222,
        "validation_file_id": upload_receipt["validation"]["azure_file_id"],
        "validation_sha256": upload_receipt["validation"]["sha256"],
        "validation_records": 85,
        "n_epochs": 2,
        "seed": 42,
        "batch_size": "automatic",
        "learning_rate_multiplier": "automatic",
        "suffix": "domain-adapt-sft-v1",
    }
    canonical_contract = json.dumps(
        job_contract, sort_keys=True, separators=(",", ":")
    )
    contract_sha256 = sha256(
        canonical_contract.encode("utf-8")
    ).hexdigest()
    assert contract_sha256 == EXPECTED_CONTRACT_SHA256
    print(json.dumps(job_contract, indent=2))
    print("Contract SHA-256:", contract_sha256)
else:
    job_contract = None
    print("Contract unavailable until the upload receipt is present.")


In [ ]:
SUBMIT_FINE_TUNING_JOB = False

if job_receipt_path.exists():
    job_receipt = json.loads(
        job_receipt_path.read_text(encoding="utf-8")
    )
    job_id = job_receipt["azure_job"]["id"]
    print("Using existing job:", job_id)
elif SUBMIT_FINE_TUNING_JOB:
    assert job_contract is not None
    fine_tuning_job = openai_client.fine_tuning.jobs.create(
        training_file=job_contract["training_file_id"],
        validation_file=job_contract["validation_file_id"],
        model="Ministral-3B",
        suffix="domain-adapt-sft-v1",
        seed=42,
        method={
            "type": "supervised",
            "supervised": {"hyperparameters": {"n_epochs": 2}},
        },
        extra_body={"trainingType": "GlobalStandard"},
    )
    job_receipt = {
        "submitted_at_utc": datetime.now(timezone.utc).isoformat(),
        "contract_sha256": contract_sha256,
        "contract": job_contract,
        "azure_job": fine_tuning_job.model_dump(mode="json"),
    }
    save_json_atomic(job_receipt_path, job_receipt)
    job_id = fine_tuning_job.id
    print("Fine-tuning job created:", job_id)
else:
    job_receipt = None
    job_id = None
    print("Submission skipped. No existing job receipt was found.")


## 5. Observe the completed job

Monitoring and result download are read-only operations. The completed job was:

- Job ID: `ftjob-bd96ca624ae04994bab47135e9d12ebb`
- Status: `succeeded`
- Fine-tuned model: `Ministral-3B.ft-bd96ca624ae04994bab47135e9d12ebb-domain-adapt-sft-v1`
- Trained tokens: 327,712 (327,830 billed)
- Reported training loss: 1.19935

Azure returned only one aggregate metric row and no validation loss or checkpoints. That proves completion, but it is not enough to infer generalization. The downstream execution evaluation is therefore the decisive evidence.


In [ ]:
if job_id is not None:
    current_job = openai_client.fine_tuning.jobs.retrieve(job_id)
    events = openai_client.fine_tuning.jobs.list_events(
        fine_tuning_job_id=job_id,
        limit=20,
    )
    print("Job ID:", current_job.id)
    print("Status:", current_job.status)
    print("Fine-tuned model:", current_job.fine_tuned_model)
    print("Trained tokens:", current_job.trained_tokens)
    print("Error:", current_job.error)
    for event in reversed(events.data):
        print(f"- [{event.level}] {event.message}")
else:
    current_job = None
    print("Monitoring skipped: no job receipt is available.")


In [ ]:
results_path = RUN_DIR / "results" / "ministral_3b_sft_v1_results.csv"
DOWNLOAD_RESULTS = False

if results_path.exists():
    print("Using downloaded metrics:", results_path)
elif DOWNLOAD_RESULTS:
    assert current_job is not None and current_job.status == "succeeded"
    result_file_id = current_job.result_files[0]
    result_bytes = openai_client.files.content(result_file_id).read()
    results_path.parent.mkdir(parents=True, exist_ok=True)
    results_path.write_bytes(result_bytes)
    print("Downloaded metrics; SHA-256:", sha256_file(results_path))
else:
    print("Metric download skipped.")

if results_path.exists():
    with results_path.open("r", encoding="utf-8-sig", newline="") as handle:
        metric_rows = list(csv.DictReader(handle))
    print("Metric rows:", len(metric_rows))
    for name in ("train_loss", "eval_loss", "eval.eval_loss"):
        values = [row.get(name) for row in metric_rows if row.get(name)]
        if values:
            print(f"{name}: {values[-1]}")


## 6. Verify the deployed tuned model

Deployment validation answers two separate questions:

1. **Connectivity:** can this client call the exact deployment name?
2. **Behavior:** can the deployment solve one known working example through the unchanged context-engineered workflow?

The Azure deployment name was truncated automatically, so the exact value in `AZURE_SFT_MODEL_DEPLOYMENT_NAME` is authoritative. Azure responses continue to report `ministral-3b-2410`; that response label identifies the model family/version and does not prove that the base deployment was called.


In [ ]:
sft_deployment_name = os.environ["AZURE_SFT_MODEL_DEPLOYMENT_NAME"].strip()
assert sft_deployment_name

RUN_CONNECTIVITY_CHECK = False
if RUN_CONNECTIVITY_CHECK:
    connection_response = openai_client.chat.completions.create(
        model=sft_deployment_name,
        messages=[{
            "role": "user",
            "content": "Reply with exactly: connection-ok",
        }],
        max_tokens=10,
    )
    print(repr(connection_response.choices[0].message.content))
    print("Response model:", connection_response.model)
else:
    print("Connectivity check skipped; deployment:", sft_deployment_name)


## 7. Reconstruct the leakage-safe working evaluation

This setup loads only the 12 `working_dev` questions. It asserts that those IDs are disjoint from the 24 protected frozen IDs. Reference-only FINCH compatibility transformations remain unchanged; generated SQL is never repaired before comparison.


In [ ]:
DB_PATH = DATA_DIR / "financial.sqlite"
DATASET_PATH = DATA_DIR / "finch_dataset.json"
MANIFEST_PATH = REPOSITORY_ROOT / "configs" / "eval_manifest_v1.json"
CONFIG_PATH = REPOSITORY_ROOT / "configs" / "slm_context_engineered_v1.json"

for required_path in (DB_PATH, DATASET_PATH, MANIFEST_PATH, CONFIG_PATH):
    if not required_path.exists():
        raise FileNotFoundError(required_path)

assert sha256_file(CONFIG_PATH) == EXPECTED_CONFIG_SHA256
context_config = load_context_engineering_config(CONFIG_PATH)
records = json.loads(DATASET_PATH.read_text(encoding="utf-8"))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))

working_ids = set(manifest["selection"]["working_dev"]["ids"])
frozen_ids = set(manifest["selection"]["frozen_eval"]["ids"])
assert working_ids.isdisjoint(frozen_ids)

working_examples = sorted(
    [
        row for row in records
        if row["db_name"] == "bird"
        and row["db_id"] == "financial"
        and row["question_id"] in working_ids
    ],
    key=lambda row: row["question_id"],
)
assert len(working_examples) == 12
assert {row["question_id"] for row in working_examples} == working_ids

conn = sqlite3.connect(
    f"file:{DB_PATH.as_posix()}?mode=ro",
    uri=True,
)

print("Working questions:", len(working_examples))
print("Frozen questions touched:", 0)


### Controlled smoke test

Question 16 passed before the batch run: both reference and candidate returned the account opening date `1993-02-26`. The optional cell below reproduces that check with one model request.


In [ ]:
RUN_QUESTION_16_SMOKE = False

if RUN_QUESTION_16_SMOKE:
    example = next(row for row in working_examples if row["question_id"] == 16)
    response, raw_sql, normalized_sql, was_unwrapped = (
        request_context_engineered_sql(
            client=openai_client,
            deployment_name=sft_deployment_name,
            question=example["question"],
            config=context_config,
        )
    )
    reference_sql, rules = make_reference_compatible(example["SQL"])
    comparison = compare_sql_results(conn, reference_sql, normalized_sql)
    print("Generated SQL:\n", normalized_sql)
    print("Passed:", comparison.equivalent)
    print("Reason:", comparison.reason)
    print("Reference-only rules:", rules)
else:
    print("Question 16 smoke test skipped.")


## 8. Run the 12-question model-selection gate

The batch runner is resumable and saves after every question. It skips IDs already present in the artifact, preventing duplicate requests after a kernel restart. It makes one attempt per question and records API failures instead of retrying them.

The first local artifact contained only immediate infrastructure errors and no model responses. It is preserved as a failed operational attempt. The valid evaluation is the `_v2` artifact below.


In [ ]:
WORKING_RUN_PATH = (
    RUN_DIR / "ministral_3b_sft_v1_working_dev_v2.json"
)
RUN_WORKING_EVALUATION = False

run_metadata = {
    "run_version": 2,
    "benchmark": manifest["benchmark"],
    "split": "working_dev",
    "variant": "ministral-3b-sft-v1",
    "deployment_name": sft_deployment_name,
    "config_sha256": EXPECTED_CONFIG_SHA256,
    "instructions_sha256": context_config["instructions_sha256"],
    "schema_context_sha256": context_config["schema_context_sha256"],
    "attempts_per_question": 1,
    "candidate_sql_repair": False,
    "retry_incorrect_answers": False,
    "output_normalization": context_config["output_normalization"],
}

if WORKING_RUN_PATH.exists():
    working_run = json.loads(
        WORKING_RUN_PATH.read_text(encoding="utf-8")
    )
    for key, expected_value in run_metadata.items():
        assert working_run[key] == expected_value
    index_results(working_run["results"])
    print("Resuming saved results:", len(working_run["results"]))
elif RUN_WORKING_EVALUATION:
    working_run = {
        **run_metadata,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
        "results": [],
    }
    save_json_atomic(WORKING_RUN_PATH, working_run)
    print("Initialized a new working run.")
else:
    working_run = None
    print("Working evaluation skipped; no local artifact found.")


In [ ]:
if RUN_WORKING_EVALUATION:
    assert working_run is not None

    for example in working_examples:
        question_id = example["question_id"]
        completed_ids = set(index_results(working_run["results"]))
        if question_id in completed_ids:
            continue

        assert question_id not in frozen_ids
        started = perf_counter()

        try:
            response, raw_sql, normalized_sql, was_unwrapped = (
                request_context_engineered_sql(
                    client=openai_client,
                    deployment_name=sft_deployment_name,
                    question=example["question"],
                    config=context_config,
                )
            )
            reference_sql, rules = make_reference_compatible(example["SQL"])
            comparison = compare_sql_results(
                conn, reference_sql, normalized_sql
            )
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": raw_sql,
                "generated_sql": normalized_sql,
                "output_envelope_normalized": was_unwrapped,
                "passed": comparison.equivalent,
                "reason": comparison.reason,
                "expected_row_count": comparison.expected_row_count,
                "candidate_row_count": comparison.candidate_row_count,
                "reference_compatibility_rules": rules,
                "response_id": response.id,
                "response_model": response.model,
                "finish_reason": response.choices[0].finish_reason,
                "usage": response.usage.model_dump() if response.usage else {},
                "api_error": None,
            }
        except Exception as error:
            result = {
                "question_id": question_id,
                "difficulty": example["difficulty"],
                "raw_generated_sql": None,
                "generated_sql": None,
                "output_envelope_normalized": False,
                "passed": False,
                "reason": "model request failed",
                "expected_row_count": None,
                "candidate_row_count": None,
                "reference_compatibility_rules": None,
                "response_id": None,
                "response_model": None,
                "finish_reason": None,
                "usage": {},
                "api_error": f"{type(error).__name__}: {error}",
            }

        result["latency_seconds"] = round(perf_counter() - started, 3)
        working_run["results"].append(result)
        index_results(working_run["results"])
        save_json_atomic(WORKING_RUN_PATH, working_run)

        outcome = "PASS" if result["passed"] else "FAIL"
        print(
            f"[{len(working_run['results']):02d}/12] "
            f"ID {question_id:>3} ({example['difficulty']:<6}) "
            f"{outcome}: {result['reason']}"
        )
else:
    print("No model requests made.")


## 9. Summarize the saved run deterministically

Accuracy, executability, failure categories, usage, and latency are properties of the saved per-question records. They do not require another model call. The reusable `evaluation_analysis` module keeps this logic identical across future notebooks.


In [ ]:
if WORKING_RUN_PATH.exists():
    working_run = json.loads(
        WORKING_RUN_PATH.read_text(encoding="utf-8")
    )
    results = working_run["results"]
    assert len(results) == 12
    assert set(index_results(results)) == working_ids

    working_summary = summarize_results(results)
    working_run["summary"] = working_summary
    save_json_atomic(WORKING_RUN_PATH, working_run)

    print(
        f"Overall: {working_summary['passed']}/12 "
        f"({working_summary['accuracy']:.1%})"
    )
    print(
        f"Executable: {working_summary['executable']}/12 "
        f"({working_summary['executable_rate']:.1%})"
    )
    print("By difficulty:", working_summary["by_difficulty"])
    print("Failure categories:", working_summary["failure_categories"])
    print("API errors:", working_summary["api_errors"])
    print("Token usage:", working_summary["token_usage"])
    print("Median latency:", working_summary["median_latency_seconds"])
    print("Artifact SHA-256:", sha256_file(WORKING_RUN_PATH))
else:
    working_summary = None
    print("Working artifact is not available locally.")


### Recorded SFT v1 working result

| Metric | Result |
|---|---:|
| Accuracy | 4/12 (33.3%) |
| Executable | 9/12 (75.0%) |
| Easy / medium / hard | 3/6, 1/4, 0/2 |
| API errors | 0 |
| Prompt / completion / total tokens | 16,406 / 2,648 / 19,054 |
| Median latency | 0.938 seconds |
| Passing IDs | 5, 16, 69, 99 |
| Artifact SHA-256 | `23cc1ffe2db857ea418d70518082c3ab782a3d15cc1f6a6c27653c6a0423297b` |

The tuned model did not discover a new correct behavior on the selection split.


## 10. Compare tuned and untuned runs

The fair comparison uses the exact same 12 question IDs and the selected Milestone 5 baseline artifact. A transition table distinguishes genuine improvements and regressions from questions that simply remained unchanged.


In [ ]:
BASELINE_RUN_PATH = (
    DATA_DIR
    / "runs"
    / "ministral_3b_compact_guardrails_strict_unwrap_working_dev_v1.json"
)

if BASELINE_RUN_PATH.exists() and WORKING_RUN_PATH.exists():
    baseline_run = json.loads(
        BASELINE_RUN_PATH.read_text(encoding="utf-8")
    )
    sft_run = json.loads(WORKING_RUN_PATH.read_text(encoding="utf-8"))
    run_comparison = compare_runs(
        baseline_run["results"],
        sft_run["results"],
    )

    print("ID | Base pass/exec | SFT pass/exec | Transition")
    print("-" * 61)
    for row in run_comparison:
        print(
            f"{row['question_id']:>3} | "
            f"{str(row['baseline_passed']):<5}/"
            f"{str(row['baseline_executable']):<5} | "
            f"{str(row['candidate_passed']):<5}/"
            f"{str(row['candidate_executable']):<5} | "
            f"{row['transition']}"
        )
else:
    run_comparison = []
    print("One or both comparison artifacts are unavailable locally.")


In [ ]:
if run_comparison:
    baseline_by_id = index_results(baseline_run["results"])
    sft_by_id = index_results(sft_run["results"])

    for row in run_comparison:
        if row["candidate_passed"]:
            continue
        question_id = row["question_id"]
        print("=" * 88)
        print("Question ID:", question_id)
        print("Transition:", row["transition"])
        print("Baseline SQL:\n", baseline_by_id[question_id].get("generated_sql"))
        print("Baseline result:", row["baseline_reason"])
        print("Tuned SQL:\n", sft_by_id[question_id].get("generated_sql"))
        print("Tuned result:", row["candidate_reason"])
else:
    print("Residual comparison skipped.")


## 11. Decision and interpretation

| Metric | Untuned context-engineered | SFT v1 | Change |
|---|---:|---:|---:|
| Accuracy | 4/12 | 4/12 | 0 |
| Executable | 8/12 | 9/12 | +1 |
| Passing IDs | 5, 16, 69, 99 | 5, 16, 69, 99 | No change |
| Completion tokens | 784 | 2,648 | +1,864 |

### What the evidence supports

- The fine-tuning service worked and produced a callable deployment.
- SFT modestly improved syntactic executability.
- It did not improve answer accuracy or solve a new working question.
- The remaining failures still include projection, ordering/semantic, join-path, and ambiguous-column errors.
- Higher completion-token use makes SFT v1 less attractive operationally as well as no more accurate.

### Gate decision

The adaptation hypothesis required a measurable improvement on `working_dev` before spending the one protected frozen evaluation. That criterion was not met. Therefore:

1. **Do not run SFT v1 on `frozen_eval`.**
2. Treat SFT v1 as a completed negative experiment, not as a deployment candidate.
3. Before considering SFT v2, inspect whether the curated cross-domain examples actually teach the specific residual behaviors and whether noisy targets diluted the signal.
4. Attempt SFT v2 only with a materially different, evidence-backed data design—not merely more epochs on the same data.

The transferable lesson is simple: training completion is an infrastructure result; behavior change is an evaluation result. Only the latter justifies promotion.
